# Load modules and data

In [1]:
from category_encoders.target_encoder import TargetEncoder
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder, LabelBinarizer, OneHotEncoder
from tqdm import tqdm
from IPython.display import display

Load data and have a look at it.

In [26]:
data = pd.read_csv('./bpic2012_O_ACCEPTED-COMPLETE.csv', ";").drop(["month", "weekday", "hour"], axis=1)
display(data.head(5))
display(data.describe())
display(data.dtypes)

,AMOUNT_REQ,Case ID,label,Activity,Resource,lifecycle:transition,timesincemidnight,timesincelastevent,timesincecasestart,event_nr,open_cases,Complete Timestamp
0,20000,173688,deviant,A_SUBMITTED-COMPLETE,112.0,COMPLETE,98,0.000000,0.000000,1,1,2011-10-01 01:38:44.546
1,20000,173688,deviant,A_PARTLYSUBMITTED-COMPLETE,112.0,COMPLETE,98,0.005567,0.005567,2,1,2011-10-01 01:38:44.880
2,20000,173688,deviant,A_PREACCEPTED-COMPLETE,112.0,COMPLETE,99,0.883767,0.889333,3,1,2011-10-01 01:39:37.906
3,20000,173688,deviant,W_Completeren aanvraag-SCHEDULE,112.0,SCHEDULE,99,0.016150,0.905483,4,1,2011-10-01 01:39:38.875
4,20000,173688,deviant,W_Completeren aanvraag-START,112.0,START,756,657.126033,658.031517,5,10,2011-10-01 12:36:46.437


,AMOUNT_REQ,Case ID,timesincemidnight,timesincelastevent,timesincecasestart,event_nr,open_cases
count,186693.000000,186693.000000,186693.000000,186693.000000,186693.000000,186693.000000,186693.000000
mean,16177.083522,192725.763971,923.109940,674.258665,13460.163161,24.573155,713.349815
std,12062.724916,11238.295836,222.632142,2353.324076,16083.528019,19.410468,150.398122
min,25.000000,173688.000000,0.000000,0.000000,0.000000,1.000000,1.000000
25%,7200.000000,182560.000000,736.000000,0.016017,197.460183,10.000000,665.000000
50%,12500.000000,193008.000000,913.000000,0.692267,9899.744133,21.000000,723.000000
75%,21000.000000,202182.000000,1093.000000,12.868517,21199.119733,33.000000,833.000000
max,99000.000000,214361.000000,1439.000000,148104.412783,197538.933533,175.000000,907.000000


AMOUNT_REQ                int64
Case ID                   int64
label                    object
Activity                 object
Resource                 object
lifecycle:transition     object
timesincemidnight         int64
timesincelastevent      float64
timesincecasestart      float64
event_nr                  int64
open_cases                int64
Complete Timestamp       object
dtype: object

# Preprocessing

## Examples of how NOT to do preprocessing! 
Loops take an awful lot of time.

In [4]:
# Manual conversion by looping through rows
for idx in tqdm(range(len(data[:1000]))):
    if data.loc[idx, "label"] == "deviant":
        data.loc[idx, "label"] = 0
    else:
        data.loc[idx, "label"] = 1
        
data

100%|██████████| 1000/1000 [00:02<00:00, 393.52it/s]


,AMOUNT_REQ,Case ID,label,Activity,Resource,lifecycle:transition,timesincemidnight,timesincelastevent,timesincecasestart,event_nr,open_cases,Complete Timestamp
0,20000,173688,0,A_SUBMITTED-COMPLETE,112.0,COMPLETE,98,0.000000,0.000000,1,1,2011-10-01 01:38:44.546
1,20000,173688,0,A_PARTLYSUBMITTED-COMPLETE,112.0,COMPLETE,98,0.005567,0.005567,2,1,2011-10-01 01:38:44.880
2,20000,173688,0,A_PREACCEPTED-COMPLETE,112.0,COMPLETE,99,0.883767,0.889333,3,1,2011-10-01 01:39:37.906
3,20000,173688,0,W_Completeren aanvraag-SCHEDULE,112.0,SCHEDULE,99,0.016150,0.905483,4,1,2011-10-01 01:39:38.875
4,20000,173688,0,W_Completeren aanvraag-START,112.0,START,756,657.126033,658.031517,5,10,2011-10-01 12:36:46.437
...,...,...,...,...,...,...,...,...,...,...,...,...
186688,5000,214361,regular,W_Valideren aanvraag-COMPLETE,10809.0,COMPLETE,1032,0.436100,16860.944317,24,240,2012-03-12 17:12:34.634
186689,5000,214361,regular,W_Valideren aanvraag-START,10138.0,START,723,1130.428533,17991.372850,25,204,2012-03-13 12:03:00.346
186690,5000,214361,regular,A_DECLINED-COMPLETE,10138.0,COMPLETE,741,0.000000,18010.069217,26,202,2012-03-13 12:21:42.128
186691,5000,214361,regular,O_DECLINED-COMPLETE,10138.0,COMPLETE,741,18.696367,18010.069217,27,202,2012-03-13 12:21:42.128


Fresh start

In [6]:
# Features are dropped for teaching purposes
# data = pd.read_csv('./BPIC2012_ACCEPTED.csv', ";").drop(["month", "weekday", "hour"], axis=1)

In [7]:
display(f" Unique values are {data['lifecycle:transition'].unique()}")
display(data.loc[:, ["Activity", "lifecycle:transition"]])

" Unique values are ['COMPLETE' 'SCHEDULE' 'START']"

,Activity,lifecycle:transition
0,A_SUBMITTED-COMPLETE,COMPLETE
1,A_PARTLYSUBMITTED-COMPLETE,COMPLETE
2,A_PREACCEPTED-COMPLETE,COMPLETE
3,W_Completeren aanvraag-SCHEDULE,SCHEDULE
4,W_Completeren aanvraag-START,START
...,...,...
186688,W_Valideren aanvraag-COMPLETE,COMPLETE
186689,W_Valideren aanvraag-START,START
186690,A_DECLINED-COMPLETE,COMPLETE
186691,O_DECLINED-COMPLETE,COMPLETE


## Still not the best approach, but way faster!
A better way, which is still quite manual if you have to deal with a lot of variables with lots of different categories.

In [8]:
data.loc[data["lifecycle:transition"] == "COMPLETE", "lifecycle:transition"] = 0
data.loc[data["lifecycle:transition"] == "SCHEDULE", "lifecycle:transition"] = 1
data.loc[data["lifecycle:transition"] == "START", "lifecycle:transition"] = 2
display(data.loc[:, ["Activity", "lifecycle:transition"]])

,Activity,lifecycle:transition
0,A_SUBMITTED-COMPLETE,0
1,A_PARTLYSUBMITTED-COMPLETE,0
2,A_PREACCEPTED-COMPLETE,0
3,W_Completeren aanvraag-SCHEDULE,1
4,W_Completeren aanvraag-START,2
...,...,...
186688,W_Valideren aanvraag-COMPLETE,0
186689,W_Valideren aanvraag-START,2
186690,A_DECLINED-COMPLETE,0
186691,O_DECLINED-COMPLETE,0


In [9]:
display(f" Unique values are {data['Activity'].unique()}")

" Unique values are ['A_SUBMITTED-COMPLETE' 'A_PARTLYSUBMITTED-COMPLETE'\n 'A_PREACCEPTED-COMPLETE' 'W_Completeren aanvraag-SCHEDULE'\n 'W_Completeren aanvraag-START' 'A_ACCEPTED-COMPLETE'\n 'O_SELECTED-COMPLETE' 'A_FINALIZED-COMPLETE' 'O_CREATED-COMPLETE'\n 'O_SENT-COMPLETE' 'W_Nabellen offertes-SCHEDULE'\n 'W_Completeren aanvraag-COMPLETE' 'W_Nabellen offertes-START'\n 'W_Nabellen offertes-COMPLETE' 'O_SENT_BACK-COMPLETE'\n 'W_Valideren aanvraag-SCHEDULE' 'W_Valideren aanvraag-START'\n 'A_REGISTERED-COMPLETE' 'A_APPROVED-COMPLETE' 'O_ACCEPTED-COMPLETE'\n 'A_ACTIVATED-COMPLETE' 'W_Valideren aanvraag-COMPLETE'\n 'O_CANCELLED-COMPLETE' 'W_Wijzigen contractgegevens-SCHEDULE'\n 'A_DECLINED-COMPLETE' 'O_DECLINED-COMPLETE'\n 'W_Nabellen incomplete dossiers-SCHEDULE'\n 'W_Nabellen incomplete dossiers-START'\n 'W_Nabellen incomplete dossiers-COMPLETE' 'W_Afhandelen leads-SCHEDULE'\n 'W_Afhandelen leads-START' 'W_Afhandelen leads-COMPLETE'\n 'A_CANCELLED-COMPLETE' 'W_Beoordelen fraude-SCHEDULE

## Proper way: Onehot Encoding with Sklearn
Sklearn has a lot of very useful preprocessing functionalities that you can and should use.

In [10]:
oh_enc = OneHotEncoder()
oh_enc = oh_enc.fit(data['Activity'].values.reshape(-1, 1))
oh_enc.transform(data['Activity'].values.reshape(-1, 1)).toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [11]:
display(oh_enc.categories_)

[array(['A_ACCEPTED-COMPLETE', 'A_ACTIVATED-COMPLETE',
        'A_APPROVED-COMPLETE', 'A_CANCELLED-COMPLETE',
        'A_DECLINED-COMPLETE', 'A_FINALIZED-COMPLETE',
        'A_PARTLYSUBMITTED-COMPLETE', 'A_PREACCEPTED-COMPLETE',
        'A_REGISTERED-COMPLETE', 'A_SUBMITTED-COMPLETE',
        'O_ACCEPTED-COMPLETE', 'O_CANCELLED-COMPLETE',
        'O_CREATED-COMPLETE', 'O_DECLINED-COMPLETE', 'O_SELECTED-COMPLETE',
        'O_SENT-COMPLETE', 'O_SENT_BACK-COMPLETE',
        'W_Afhandelen leads-COMPLETE', 'W_Afhandelen leads-SCHEDULE',
        'W_Afhandelen leads-START', 'W_Beoordelen fraude-COMPLETE',
        'W_Beoordelen fraude-SCHEDULE', 'W_Beoordelen fraude-START',
        'W_Completeren aanvraag-COMPLETE',
        'W_Completeren aanvraag-SCHEDULE', 'W_Completeren aanvraag-START',
        'W_Nabellen incomplete dossiers-COMPLETE',
        'W_Nabellen incomplete dossiers-SCHEDULE',
        'W_Nabellen incomplete dossiers-START',
        'W_Nabellen offertes-COMPLETE', 'W_Nabellen offer

## Best approach
Sklearn is sometimes quite clunky to use. Pandas also provides a very quick and less complicated way to turn categories into one hot encodings. 
Don't forget to drop the first column.

In [12]:
dummies = pd.get_dummies(data['Activity'], drop_first=True)
data.join(dummies).drop('Activity', axis=1)

,AMOUNT_REQ,Case ID,label,Resource,lifecycle:transition,timesincemidnight,timesincelastevent,timesincecasestart,event_nr,open_cases,...,W_Nabellen incomplete dossiers-COMPLETE,W_Nabellen incomplete dossiers-SCHEDULE,W_Nabellen incomplete dossiers-START,W_Nabellen offertes-COMPLETE,W_Nabellen offertes-SCHEDULE,W_Nabellen offertes-START,W_Valideren aanvraag-COMPLETE,W_Valideren aanvraag-SCHEDULE,W_Valideren aanvraag-START,W_Wijzigen contractgegevens-SCHEDULE
0,20000,173688,0,112.0,0,98,0.000000,0.000000,1,1,...,0,0,0,0,0,0,0,0,0,0
1,20000,173688,0,112.0,0,98,0.005567,0.005567,2,1,...,0,0,0,0,0,0,0,0,0,0
2,20000,173688,0,112.0,0,99,0.883767,0.889333,3,1,...,0,0,0,0,0,0,0,0,0,0
3,20000,173688,0,112.0,1,99,0.016150,0.905483,4,1,...,0,0,0,0,0,0,0,0,0,0
4,20000,173688,0,112.0,2,756,657.126033,658.031517,5,10,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186688,5000,214361,regular,10809.0,0,1032,0.436100,16860.944317,24,240,...,0,0,0,0,0,0,1,0,0,0
186689,5000,214361,regular,10138.0,2,723,1130.428533,17991.372850,25,204,...,0,0,0,0,0,0,0,0,1,0
186690,5000,214361,regular,10138.0,0,741,0.000000,18010.069217,26,202,...,0,0,0,0,0,0,0,0,0,0
186691,5000,214361,regular,10138.0,0,741,18.696367,18010.069217,27,202,...,0,0,0,0,0,0,0,0,0,0


## Label encoding
For classification task you might want to turn a categorical label-column with strings into a categorical label-column with numbers. Sklearn has some functionality for that.

In [29]:
lbl_enc = LabelEncoder()
lbl_enc = lbl_enc.fit(data["label"])
data["label"] = lbl_enc.transform(data["label"])
data["label"]

0         0
1         0
2         0
3         0
4         0
         ..
186688    1
186689    1
186690    1
186691    1
186692    1
Name: label, Length: 186693, dtype: int64

## Type Conversion

How to turn non-numerical variables into numerical ones. Important requirement is that the column does not have any non-numerical values in it's column. So we first turn other into "0".

In [15]:
data.Resource[data.Resource == "other"] = "0"
data.Resource = pd.to_numeric(data.Resource).astype(int)
data.dtypes

/Users/xixilu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


AMOUNT_REQ                int64
Case ID                   int64
label                    object
Activity                 object
Resource                  int64
lifecycle:transition     object
timesincemidnight         int64
timesincelastevent      float64
timesincecasestart      float64
event_nr                  int64
open_cases                int64
Complete Timestamp       object
dtype: object

## Extract Time features
First convert the column into a datetime column.

In [16]:
from datetime import datetime

data["Complete Timestamp"] = pd.to_datetime(data["Complete Timestamp"])
data.dtypes

AMOUNT_REQ                       int64
Case ID                          int64
label                           object
Activity                        object
Resource                         int64
lifecycle:transition            object
timesincemidnight                int64
timesincelastevent             float64
timesincecasestart             float64
event_nr                         int64
open_cases                       int64
Complete Timestamp      datetime64[ns]
dtype: object

Now you can extract various information from the time stamp.

In [17]:
data["Complete Timestamp"].dt.week
data["Complete Timestamp"].dt.day
data["Complete Timestamp"].dt.month

/Users/xixilu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: FutureWarning: Series.dt.weekofyear and Series.dt.week have been deprecated.  Please use Series.dt.isocalendar().week instead.
  """Entry point for launching an IPython kernel.


0         10
1         10
2         10
3         10
4         10
          ..
186688     3
186689     3
186690     3
186691     3
186692     3
Name: Complete Timestamp, Length: 186693, dtype: int64

You can even create new features.

In [18]:
data["month"] = data["Complete Timestamp"].dt.month
data["week"] = data["Complete Timestamp"].dt.week
data["day"] = data["Complete Timestamp"].dt.day
data = data.drop(["Complete Timestamp"], axis=1)
data.head()

/Users/xixilu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:2: FutureWarning: Series.dt.weekofyear and Series.dt.week have been deprecated.  Please use Series.dt.isocalendar().week instead.
  


,AMOUNT_REQ,Case ID,label,Activity,Resource,lifecycle:transition,timesincemidnight,timesincelastevent,timesincecasestart,event_nr,open_cases,month,week,day
0,20000,173688,0,A_SUBMITTED-COMPLETE,112,0,98,0.000000,0.000000,1,1,10,39,1
1,20000,173688,0,A_PARTLYSUBMITTED-COMPLETE,112,0,98,0.005567,0.005567,2,1,10,39,1
2,20000,173688,0,A_PREACCEPTED-COMPLETE,112,0,99,0.883767,0.889333,3,1,10,39,1
3,20000,173688,0,W_Completeren aanvraag-SCHEDULE,112,1,99,0.016150,0.905483,4,1,10,39,1
4,20000,173688,0,W_Completeren aanvraag-START,112,2,756,657.126033,658.031517,5,10,10,39,1


## Normalizing your data
Sklearn has lots of scalers. You should read up on them. They are simple to use.

In [19]:
scaler = MinMaxScaler()
date_vars = ["month", "week", "day"]
scaler.fit(data[date_vars])
data[date_vars] = scaler.transform(data[date_vars])
data[["Case ID", "Activity"] + date_vars].describe()

,Case ID,month,week,day
count,186693.000000,186693.000000,186693.000000,186693.000000
mean,192725.763971,0.530152,0.526982,0.482137
std,11238.295836,0.433781,0.405852,0.293422
min,173688.000000,0.000000,0.000000,0.000000
25%,182560.000000,0.090909,0.098039,0.233333
50%,193008.000000,0.818182,0.784314,0.466667
75%,202182.000000,0.909091,0.901961,0.733333
max,214361.000000,1.000000,1.000000,1.000000


## Previous Activity Encoding

In [20]:
data['prev_activity'] = np.roll(data["Activity"], 1)
data[["Case ID", "Activity", "prev_activity"]]

,Case ID,Activity,prev_activity
0,173688,A_SUBMITTED-COMPLETE,W_Valideren aanvraag-COMPLETE
1,173688,A_PARTLYSUBMITTED-COMPLETE,A_SUBMITTED-COMPLETE
2,173688,A_PREACCEPTED-COMPLETE,A_PARTLYSUBMITTED-COMPLETE
3,173688,W_Completeren aanvraag-SCHEDULE,A_PREACCEPTED-COMPLETE
4,173688,W_Completeren aanvraag-START,W_Completeren aanvraag-SCHEDULE
...,...,...,...
186688,214361,W_Valideren aanvraag-COMPLETE,W_Valideren aanvraag-START
186689,214361,W_Valideren aanvraag-START,W_Valideren aanvraag-COMPLETE
186690,214361,A_DECLINED-COMPLETE,W_Valideren aanvraag-START
186691,214361,O_DECLINED-COMPLETE,A_DECLINED-COMPLETE


Remove shifting overlaps

In [21]:
data.loc[data["Activity"] == "A_SUBMITTED-COMPLETE", "prev_activity"] = None
data[["Case ID", "Activity", "prev_activity"]].head(50)

,Case ID,Activity,prev_activity
0,173688,A_SUBMITTED-COMPLETE,None
1,173688,A_PARTLYSUBMITTED-COMPLETE,A_SUBMITTED-COMPLETE
2,173688,A_PREACCEPTED-COMPLETE,A_PARTLYSUBMITTED-COMPLETE
3,173688,W_Completeren aanvraag-SCHEDULE,A_PREACCEPTED-COMPLETE
4,173688,W_Completeren aanvraag-START,W_Completeren aanvraag-SCHEDULE
5,173688,A_ACCEPTED-COMPLETE,W_Completeren aanvraag-START
6,173688,O_SELECTED-COMPLETE,A_ACCEPTED-COMPLETE
7,173688,A_FINALIZED-COMPLETE,O_SELECTED-COMPLETE
8,173688,O_CREATED-COMPLETE,A_FINALIZED-COMPLETE
9,173688,O_SENT-COMPLETE,O_CREATED-COMPLETE


In [22]:
data.loc[data["Case ID"] == 214361, ["Case ID", "Activity", "prev_activity"]]

,Case ID,Activity,prev_activity
186665,214361,A_SUBMITTED-COMPLETE,None
186666,214361,A_PARTLYSUBMITTED-COMPLETE,A_SUBMITTED-COMPLETE
186667,214361,A_PREACCEPTED-COMPLETE,A_PARTLYSUBMITTED-COMPLETE
186668,214361,W_Completeren aanvraag-SCHEDULE,A_PREACCEPTED-COMPLETE
186669,214361,W_Completeren aanvraag-START,W_Completeren aanvraag-SCHEDULE
186670,214361,W_Completeren aanvraag-COMPLETE,W_Completeren aanvraag-START
186671,214361,W_Completeren aanvraag-START,W_Completeren aanvraag-COMPLETE
186672,214361,A_ACCEPTED-COMPLETE,W_Completeren aanvraag-START
186673,214361,A_FINALIZED-COMPLETE,A_ACCEPTED-COMPLETE
186674,214361,O_SELECTED-COMPLETE,A_FINALIZED-COMPLETE


## An alternative case-by-case way of doing the same

In [23]:
mod_df = pd.DataFrame()
for idx, df in data[:100].groupby("Case ID"):
    # display(df)
    df = df.sort_values("event_nr")
    df['prev_activity'] = np.roll(df["Activity"], 1)
    df.iloc[0, df.columns == "prev_activity"] = None
    mod_df = pd.concat([mod_df, df])
mod_df.head(30)

,AMOUNT_REQ,Case ID,label,Activity,Resource,lifecycle:transition,timesincemidnight,timesincelastevent,timesincecasestart,event_nr,open_cases,month,week,day,prev_activity
0,20000,173688,0,A_SUBMITTED-COMPLETE,112,0,98,0.000000,0.000000,1,1,0.818182,0.745098,0.000000,None
1,20000,173688,0,A_PARTLYSUBMITTED-COMPLETE,112,0,98,0.005567,0.005567,2,1,0.818182,0.745098,0.000000,A_SUBMITTED-COMPLETE
2,20000,173688,0,A_PREACCEPTED-COMPLETE,112,0,99,0.883767,0.889333,3,1,0.818182,0.745098,0.000000,A_PARTLYSUBMITTED-COMPLETE
3,20000,173688,0,W_Completeren aanvraag-SCHEDULE,112,1,99,0.016150,0.905483,4,1,0.818182,0.745098,0.000000,A_PREACCEPTED-COMPLETE
4,20000,173688,0,W_Completeren aanvraag-START,112,2,756,657.126033,658.031517,5,10,0.818182,0.745098,0.000000,W_Completeren aanvraag-SCHEDULE
5,20000,173688,0,A_ACCEPTED-COMPLETE,10862,0,762,5.947850,663.979367,6,10,0.818182,0.745098,0.000000,W_Completeren aanvraag-START
6,20000,173688,0,O_SELECTED-COMPLETE,10862,0,765,0.000000,666.411617,7,10,0.818182,0.745098,0.000000,A_ACCEPTED-COMPLETE
7,20000,173688,0,A_FINALIZED-COMPLETE,10862,0,765,2.432250,666.411617,8,10,0.818182,0.745098,0.000000,O_SELECTED-COMPLETE
8,20000,173688,0,O_CREATED-COMPLETE,10862,0,765,0.032567,666.444183,9,10,0.818182,0.745098,0.000000,A_FINALIZED-COMPLETE
9,20000,173688,0,O_SENT-COMPLETE,10862,0,765,0.003050,666.447233,10,10,0.818182,0.745098,0.000000,O_CREATED-COMPLETE


## Bringing all together!
This is a full preprocessing pipeline if it is the goal to predict the label.

In [31]:
# Load data
# data = pd.read_csv('./BPIC2012_ACCEPTED.csv', sep=";").drop(["month", "weekday", "hour"], axis=1)

# Add Prior Activity
data['prev_activity'] = np.roll(data["Activity"], 1)
data['prev_prev_activity'] = np.roll(data["Activity"], 2)
data.loc[data["event_nr"] == 1, ["prev_activity", "prev_prev_activity"]] = None
data.loc[data["event_nr"] == 2, "prev_prev_activity"] = None

# Extract Time Variables
data["Complete Timestamp"] = pd.to_datetime(data["Complete Timestamp"])
data["week"] = data["Complete Timestamp"].dt.week
data["month"] = data["Complete Timestamp"].dt.month
data["day"] = data["Complete Timestamp"].dt.day
data["hour"] = data["Complete Timestamp"].dt.hour
# data["hour"] = data["Complete Timestamp"].dt.hour
data = data.drop(["Complete Timestamp"], axis=1)

# Transform Label
lbl_enc = LabelEncoder()
lbl_enc = lbl_enc.fit(data["label"])
data["label"] = lbl_enc.transform(data["label"])

# Change Resource column
data.Resource[data.Resource == "other"] = "0"
data.Resource = pd.to_numeric(data.Resource).astype(int).astype(str)

# Dummify data
columns_to_dummy_encode = ["Resource", "Activity", "prev_activity", "prev_prev_activity", "lifecycle:transition"]
dummyfied_data = pd.get_dummies(data[columns_to_dummy_encode], drop_first=True)
data = data.join(dummyfied_data)
data = data.drop(columns_to_dummy_encode, axis=1)

# Change the index
data.index = data["Case ID"]
data = data.drop("Case ID", axis=1)

# Normalize Data
scaler = MinMaxScaler()
normalize_vars = [
    "AMOUNT_REQ",
    "open_cases",
    "event_nr",
    "timesincemidnight",
    "timesincelastevent",
    "timesincecasestart",
    "month",
    "week",
    "day",
]
scaler.fit(data[normalize_vars])
data[normalize_vars] = scaler.transform(data[normalize_vars])
data

/Users/xixilu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:12: FutureWarning: Series.dt.weekofyear and Series.dt.week have been deprecated.  Please use Series.dt.isocalendar().week instead.
  if sys.path[0] == '':
/Users/xixilu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


,AMOUNT_REQ,label,timesincemidnight,timesincelastevent,timesincecasestart,event_nr,open_cases,week,month,day,...,prev_prev_activity_W_Nabellen incomplete dossiers-START,prev_prev_activity_W_Nabellen offertes-COMPLETE,prev_prev_activity_W_Nabellen offertes-SCHEDULE,prev_prev_activity_W_Nabellen offertes-START,prev_prev_activity_W_Valideren aanvraag-COMPLETE,prev_prev_activity_W_Valideren aanvraag-SCHEDULE,prev_prev_activity_W_Valideren aanvraag-START,prev_prev_activity_W_Wijzigen contractgegevens-SCHEDULE,lifecycle:transition_SCHEDULE,lifecycle:transition_START
Case ID,,,,,,,,,,,,,,,,,,,,,
173688,0.201819,0,0.068103,0.000000e+00,0.000000e+00,0.000000,0.000000,0.745098,0.818182,0.000000,...,0,0,0,0,0,0,0,0,0,0
173688,0.201819,0,0.068103,3.758610e-08,2.818010e-08,0.005747,0.000000,0.745098,0.818182,0.000000,...,0,0,0,0,0,0,0,0,0,0
173688,0.201819,0,0.068798,5.967187e-06,4.502066e-06,0.011494,0.000000,0.745098,0.818182,0.000000,...,0,0,0,0,0,0,0,0,0,0
173688,0.201819,0,0.068798,1.090447e-07,4.583822e-06,0.017241,0.000000,0.745098,0.818182,0.000000,...,0,0,0,0,0,0,0,0,1,0
173688,0.201819,0,0.525365,4.436911e-03,3.331148e-03,0.022989,0.009934,0.745098,0.818182,0.000000,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214361,0.050265,1,0.717165,2.944544e-06,8.535504e-02,0.132184,0.263797,0.196078,0.181818,0.366667,...,0,1,0,0,0,0,0,0,0,0
214361,0.050265,1,0.502432,7.632646e-03,9.107760e-02,0.137931,0.224062,0.196078,0.181818,0.400000,...,0,0,0,0,0,0,1,0,0,1
214361,0.050265,1,0.514941,0.000000e+00,9.117225e-02,0.143678,0.221854,0.196078,0.181818,0.400000,...,0,0,0,0,1,0,0,0,0,0


## Using the newly created data for classification

In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, train_test_split

# Just so that the demonstration doesn't take too long
data_subset = data.sample(100000)
n_splits = 5
n_repeats = 10
X, y = data_subset.drop("label", axis=1), data_subset["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.75, random_state=42)
cv = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=42)
clf = RandomForestClassifier()

cv_results = cross_val_score(clf, X_train, y_train, cv=cv, verbose=3, n_jobs=3)
cv_results

[Parallel(n_jobs=3)]: Using backend LokyBackend with 3 concurrent workers.


[CV] END ................................ score: (test=0.717) total time=   7.2s
[CV] END ................................ score: (test=0.707) total time=   7.6s
[CV] END ................................ score: (test=0.726) total time=   9.9s
[CV] END ................................ score: (test=0.717) total time=   8.8s
[CV] END ................................ score: (test=0.717) total time=   9.0s
[CV] END ................................ score: (test=0.713) total time=   8.6s
[CV] END ................................ score: (test=0.712) total time=   6.8s
[CV] END ................................ score: (test=0.716) total time=   9.3s
[CV] END ................................ score: (test=0.710) total time=   9.2s
[CV] END ................................ score: (test=0.721) total time=   9.2s
[CV] END ................................ score: (test=0.700) total time=   9.5s
[CV] END ................................ score: (test=0.701) total time=   9.4s
[CV] END ...................

[Parallel(n_jobs=3)]: Done  26 tasks      | elapsed:  1.4min


[CV] END ................................ score: (test=0.722) total time=   9.5s
[CV] END ................................ score: (test=0.714) total time=   7.8s
[CV] END ................................ score: (test=0.709) total time=   8.1s
[CV] END ................................ score: (test=0.709) total time=  10.1s
[CV] END ................................ score: (test=0.707) total time=   9.9s
[CV] END ................................ score: (test=0.719) total time=  11.0s
[CV] END ................................ score: (test=0.726) total time=  10.9s
[CV] END ................................ score: (test=0.714) total time=  12.1s
[CV] END ................................ score: (test=0.703) total time=   9.1s
[CV] END ................................ score: (test=0.710) total time=   8.7s
[CV] END ................................ score: (test=0.721) total time=   9.0s
[CV] END ................................ score: (test=0.715) total time=   9.4s
[CV] END ...................

[Parallel(n_jobs=3)]: Done  50 out of  50 | elapsed:  2.6min finished


array([0.7258, 0.7074, 0.7166, 0.717 , 0.717 , 0.7128, 0.716 , 0.7104,
       0.7116, 0.6996, 0.7212, 0.7008, 0.7142, 0.7136, 0.7208, 0.7094,
       0.7134, 0.7106, 0.7018, 0.7144, 0.7166, 0.7128, 0.7178, 0.7106,
       0.7178, 0.7168, 0.7224, 0.7142, 0.7092, 0.707 , 0.7086, 0.7188,
       0.7256, 0.7138, 0.7026, 0.7098, 0.7214, 0.7154, 0.7196, 0.7098,
       0.7142, 0.7164, 0.7158, 0.7106, 0.7084, 0.7182, 0.7124, 0.7154,
       0.7066, 0.7038])

In [23]:
cv_results.reshape(n_splits,n_repeats).mean(axis=0).mean()

0.7133360000000001

In [24]:
# A couple of more thoughts for Next Activity Prediction
# -     What's the label we want to predict?
# -     What about the label column?
# -     What about the hour column?